# 03 — The GQE solver

Running the CUDA-Q generative quantum eigensolver against a DMET
embedding.

**This notebook needs more setup than the others**: `quenais[cudaq]`, a
`gqe-for-qsci` checkout at commit `0a201ea`, and the source patch applied.
See `docs/gqe_integration.md`.

Everything before the training cell works without CUDA-Q, so you can read
the configuration parts regardless.

### Read this before launching anything long

**`--gqe-seed` is not optional if you intend to repeat a run.**

`gqe-for-qsci/configs/trainer/default.yaml` pins `seed: 32`. `GqeSettings.seed`
defaults to `None`, which — by the override mechanism's design — means *leave
the config's value alone*, not *randomise*. So every GQE run launched without an
explicit seed trains on the identical seed and reproduces the identical
determinant set and the identical energy.

Three "repeat" runs at N₂ 1.8 Å once gave 78.9 mHa to six significant figures
each. That looks exactly like a systematic, deterministic solver bug, and it
cost several hours and three separate wrong hypotheses about *why GQE was
failing* before anyone checked whether the runs were independent at all.

```bash
# WRONG - three bit-identical runs
for i in 1 2 3; do quenais-run --solver gqe --project-dir runs/rep$i ...; done

# RIGHT
for s in 101 102 103; do
  quenais-run --solver gqe --gqe-seed $s --project-dir runs/seed$s ...
done
```

Before quoting any run-to-run spread, confirm `--gqe-seed` was actually passed
*and varied*. Full writeup: `docs/reproducibility.md` §5.

### Cost expectations

Circuit simulation and the transformer are separate costs on separate devices —
`cudaq_target` selects the **circuit simulator** only. The transformer trains on
GPU via Lightning either way, which is why a CPU-simulator run still logs
`GPU available: True, used: True`. That log line has been misread as "the GPU is
doing the circuits" more than once.

| setting | effect |
|---|---|
| `max_iters` | training epochs. Default 120. Use **2** for a smoke test |
| `ngates` | circuit depth. Default 40. On ScH: 10 stalled at HF, 20 plateaued, 40 recovered ~60% of the correlation energy |
| `num_samples` | samples per epoch (default 100). `batch_size` must equal it — `GqeSettings.validate()` enforces this |
| `qsci_max_dim` | QSCI subspace cap. Default 10000; the upstream default of 2000 became the binding constraint on ScH |
| `cudaq_target` | `nvidia` needs compute capability ≥ 8.0, else `qpp-cpu` |

Always do a `max_iters=2` smoke test against a new checkout before launching a
120-epoch run.

## 1. Is the checkout ready?

The runner refuses to launch against an unpatched checkout. Worth checking
first — the failure mode otherwise is a training run that completes
successfully and produces nothing parseable.

In [ ]:
from quenais.quantum import gqe_setup

repo = "../gqe-for-qsci"          # adjust to your checkout

problems = gqe_setup.verify_gqe_repo(repo)
if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
    print(f"\nFix with: quenais-gqe-setup --repo {repo}")
else:
    print("Ready: correct commit, patch applied.")

## 2. Configure

The GQE settings map onto the external repo's Hydra config. Anything left
as `None` uses that repo's own default.

Two overrides are mandatory and always emitted:

- `molecule=dmet_embedding` — without it, `train.py` loads `n2.yaml` and
  trains on N₂ while reporting numbers that claim to be yours.
- `operator_pool.spec=dmet_excitation` — the stock pools rebuild the
  molecule from its geometry, and an embedding has none.

That failure is not hypothetical — it is systematic error #4 in the thesis.
Every run trained on N₂ while labelled LiH or ScH, and the tell was not an
error message: it was that **two different molecules produced identical epoch
logs**, at around −107 E_h, which is N₂'s energy scale rather than either
intended system's.

Sanity check any new training run by looking at the *magnitude* of the energy in
the first few epochs. LiH should be ≈ −7.9, N₂ ≈ −107.6, ScH ≈ −752.7. If the
scale is wrong, stop — the override did not take.

In [ ]:
from quenais import Config
from quenais.settings import GqeSettings

cfg = Config(
    molecule="LiH",
    basis="sto-3g",
    project_dir="./lih_run",
    quantum_solver="gqe",
    gqe=GqeSettings(
        repo_path=repo,
        max_iters=30,          # small, for a first run
        num_samples=100,
        batch_size=100,        # must equal num_samples
        ngates=20,
        cudaq_target="qpp-cpu",
    ),
)
cfg.validate().make_dirs().load_geometry()

for override in cfg.gqe.hydra_overrides(cfg.step2_file):
    print(" ", override)

## 3. Which pool, and why it matters

`dmet_excitation` accumulates all the Pauli terms of an excitation into a
single operator. Particle-number conservation is a property of that
**sum** — no individual Pauli word conserves it alone.

`dmet_pauli_evolution` appends each term separately, so it cannot conserve
electron number however its flags are set. On ScH roughly half of every
sample was discarded as symmetry-violating.

In [ ]:
from quenais.settings.gqe import DMET_POOL_SPECS, OPERATOR_POOL_SPECS

print("registered by the patch :", OPERATOR_POOL_SPECS)
print("usable with an embedding:", DMET_POOL_SPECS)

# The stock pools are rejected at config time rather than 20 minutes into a run.
try:
    GqeSettings(operator_pool_spec="excitation").validate()
except ValueError as exc:
    print("\n", exc)

## 4. Train

Runs the external `train.py` as a subprocess, streaming its output. The
runner verifies the patch, checks the step 2 pickle belongs to this
molecule, generates the Hydra molecule config, and puts the shim directory
on `PYTHONPATH`.

LiH converges to the embedded CASCI energy essentially exactly, so it is a
clean pass/fail.

In [ ]:
from quenais.quantum import gqe_runner

result = gqe_runner.main(cfg, force=True)
print(result)

## 5. Read the training log

The `[epoch N] {...}` lines come from the `train_pipeline.py` patch hunk.
If there are none, the checkout is unpatched — training will have looked
entirely successful.

In [ ]:
from quenais.visualization import plots

rows = plots.parse_gqe_log(cfg.gqe_log_file)
print(f"{len(rows)} epochs")

if rows:
    key = "Global-refined(best_so_far)/energy - R-CASCI"
    x, y = plots._col(rows, key)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, y)
    ax.axhline(1.6e-3, color="gray", ls="--", label="chemical accuracy")
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("|error vs CASCI| (Ha)")
    ax.legend()

## 6. Figures and summary

In [ ]:
plots.main(cfg)

## Sampling capacity is the limit on larger systems

GQE's accuracy is bounded by how much of the determinant space it can
reach. ScH (22 qubits, ~109k determinants) showed this clearly:

| settings | error vs embedded CASCI |
|---|---|
| ngates=10, samples=10 | 60.4 mHa — stalled at HF |
| ngates=20, samples=100 | 36.8 mHa |
| ngates=40, samples=100 | 24.1 mHa — subspace hit the cap |

LiH and N₂ (4 embedding orbitals) converge exactly, so this is a capacity
limit on larger systems rather than a correctness problem. See
`docs/limitations.md`.

---

## Reading the epoch log

`results/gqe_epoch_log.csv` is written by the external trainer. The columns that
matter:

| column | meaning |
|---|---|
| `GQE-optimized(best_so_far)/energy - R-CASCI` | error vs the exact embedded answer — **the number you care about** |
| `.../num_sampled_basis` | distinct determinants the sampler found this epoch |
| `.../num_symmetry_preserving_basis` | how many survived the particle-number filter |
| `.../subspace_dim` | dimension actually diagonalised |
| `Global-refined`, `Local-refined` | post-hoc classical refinement of the sampled set |

**If `num_symmetry_preserving_basis` is much smaller than `num_sampled_basis`,
the pool is wrong.** That gap was ~50% on ScH before the pool was rebuilt from
whole excitations — particle-number conservation belongs to the full sum of
Pauli terms in a mapped excitation, never to a single term, so a pool built from
individual Pauli strings cannot conserve N however its flags are set.

**If `subspace_dim` equals `qsci_max_dim` exactly, the run terminated at the cap
and had not converged.** Any error you quote from it is an upper bound on a
capacity limit, not a converged result. This is exactly what happened at
`n_g=40, N_s=100` on ScH: `subspace_dim = 2000` = the cap.

## Before trusting a GQE number

1. Was `--gqe-seed` passed, and varied across repeats?
2. Is the energy scale right for the molecule you think you ran?
3. Is `subspace_dim` below the cap?
4. Is `num_symmetry_preserving_basis ≈ num_sampled_basis`?
5. Does the same run, repeated, give a *different* answer? (It should. If three
   runs agree to six figures, see the seed warning at the top.)

## The question this notebook cannot answer

A GQE error of *X* mHa means nothing on its own — you cannot tell whether the
sampler did badly or whether the problem was easy enough that anything would
have worked. `05_determinant_selection` measures that directly, against an
oracle bound and a classical baseline at identical cost. On ScH the answer
turned out to be "the problem was easy": a 1973 classical method beat GQE by
~4,900×, because ScH cannot discriminate selection methods at all.